# UniProt — Protein Sequences and Functional Annotation

**UniProt** (Universal Protein Resource) is the ELIXIR Core Data Resource for protein sequences and functional information. It is the world's most comprehensive, high-quality, and freely accessible database of protein sequences and their functional annotations.

UniProt is organised into three components:

| Component | Description | Size (approx.) |
|---|---|---|
| **UniProtKB/Swiss-Prot** | Manually reviewed, expert-curated, highest quality | ~570 k entries |
| **UniProtKB/TrEMBL** | Computationally annotated, awaiting review | ~250 M entries |
| **UniRef** | Clustered sequence sets at 100%, 90%, 50% identity | billions of clusters |
| **UniParc** | Non-redundant sequence archive across all major DBs | ~700 M sequences |

This notebook focuses on **UniProtKB/Swiss-Prot for *Homo sapiens*** — the gold-standard reviewed subset — and uses the [UniProt REST API](https://rest.uniprot.org) (v2023+).

**Reference:** The UniProt Consortium (2023), *Nucleic Acids Research*, UniProt 2023.

In [ ]:
import re
import time
from pathlib import Path

import requests
import polars as pl
import pandas as pd

# TODO

* [x] **Ingest data**
    * [x] Connect to UniProt REST API and confirm the current release version
    * [x] Fetch a single well-known entry (P04637 / human TP53) to verify connectivity
    * [x] Download all reviewed *H. sapiens* proteins from UniProtKB/Swiss-Prot in TSV format
    * [x] Handle cursor-based pagination via the `Link` response header
    * [x] Save results to `data/uniprot_human_swissprot.tsv`
    * [x] Parse TSV into a Polars DataFrame with normalised snake_case column names
* [ ] **Explore and clean**
    * [ ] Summarise dataset dimensions, dtypes, and missing-value counts
    * [ ] Inspect annotation score and evidence level distributions
    * [ ] Filter entries by protein existence evidence level (PE1 = experimental)
    * [ ] Detect and handle duplicated accessions or ambiguous gene names
* [ ] **Sequence analysis**
    * [ ] Plot sequence length distribution (log-scale histogram)
    * [ ] Compute amino acid composition and compare to background frequencies
    * [ ] Identify signal peptides, transmembrane regions, and disordered segments from feature counts
    * [ ] Examine domain/family coverage (Pfam, PROSITE cross-references)
* [ ] **Functional annotation**
    * [ ] Summarise GO term coverage (Biological Process, Molecular Function, Cellular Component)
    * [ ] Map entries to KEGG pathways and Reactome via cross-references
    * [ ] Analyse disease associations (Swiss-Prot disease annotations)
    * [ ] Assess annotation completeness across protein families
* [ ] **Visualization**
    * [ ] Subcellular localisation sunburst or treemap chart
    * [ ] Sequence feature map for a protein of interest (e.g. TP53)
    * [ ] Phylogenetic coverage heatmap (number of reviewed orthologs per taxon)
    * [ ] GO term bubble plot (semantic similarity vs. enrichment p-value)
* [ ] **Statistical analysis**
    * [ ] Model annotation completeness as a function of protein age and expression breadth
    * [ ] Test whether disease-associated proteins have higher annotation scores
    * [ ] Discuss multiple-hypothesis correction for GO enrichment analyses
    * [ ] Propagate uncertainty in annotation scores to downstream metrics

## 1. Ingest Data

### 1.1 Connect to UniProt API

In [ ]:
UNIPROT_BASE = "https://rest.uniprot.org"

# --- Fetch release version from the API ---
_release_resp = requests.get(f"{UNIPROT_BASE}/api/release-notes/latest", timeout=30)
_release_resp.raise_for_status()
UNIPROT_VERSION = _release_resp.json().get("releaseNumber", "unknown")
print(f"UniProt release: {UNIPROT_VERSION}")


def uniprot_get(endpoint: str, params: dict | None = None) -> requests.Response:
    """
    Send a GET request to the UniProt REST API with a polite 1-second delay.

    Parameters
    ----------
    endpoint : str
        API path relative to UNIPROT_BASE (e.g. ``"uniprotkb/P04637"``).
    params : dict or None, optional
        Query parameters to append to the URL. ``None`` sends no parameters.

    Returns
    -------
    requests.Response
        Raw response object with status already checked via ``raise_for_status``.

    Notes
    -----
    The 1-second sleep after every request respects UniProt's fair-use guidance
    for programmatic access. For bulk downloads, prefer the cursor-paginated
    search endpoint rather than making thousands of individual entry calls.
    """
    url = f"{UNIPROT_BASE}/{endpoint.lstrip('/')}"
    resp = requests.get(url, params=params or {}, timeout=60)
    resp.raise_for_status()
    time.sleep(1)   # polite rate-limiting
    return resp


# --- Connectivity check: fetch human TP53 (accession P04637) ---
tp53 = uniprot_get("uniprotkb/P04637", {"format": "json"}).json()
print(f"Entry: {tp53['primaryAccession']}  —  {tp53['uniProtkbId']}")
print(f"Protein: {tp53['proteinDescription']['recommendedName']['fullName']['value']}")

### 1.2 Download Human Swiss-Prot Entries

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

SWISSPROT_PATH = DATA_DIR / "uniprot_human_swissprot.tsv"

# Fields requested from the search API — covers the most analytically useful columns
FIELDS = (
    "accession,id,protein_name,gene_names,organism_name,"
    "length,mass,sequence,"
    "go_ids,go,keyword,"
    "feature_count,annotation_score,reviewed"
)

# UniProt REST uses cursor-based pagination: after the first request, each response
# carries a `Link: <url>; rel="next"` header pointing to the next page.
_LINK_RE = re.compile(r'<([^>]+)>;\s*rel="next"')


def _parse_next_link(link_header: str | None) -> str | None:
    """
    Extract the next-page URL from a UniProt ``Link`` response header.

    Parameters
    ----------
    link_header : str or None
        Raw value of the ``Link`` HTTP response header, e.g.
        ``'<https://rest.uniprot.org/uniprotkb/search?cursor=…>; rel="next"'``.

    Returns
    -------
    str or None
        Absolute URL for the next page, or ``None`` if the header is absent or
        contains no ``rel="next"`` relation (i.e. we are on the last page).
    """
    if not link_header:
        return None
    match = _LINK_RE.search(link_header)
    return match.group(1) if match else None


if SWISSPROT_PATH.exists():
    print(f"Already downloaded: {SWISSPROT_PATH}")
else:
    print("Downloading reviewed H. sapiens entries from UniProtKB …")

    # First page — subsequent pages are driven by the cursor URL in Link header
    params = {
        "query": "reviewed:true AND organism_id:9606",
        "format": "tsv",
        "fields": FIELDS,
        "size": 500,   # max page size allowed by the API
    }

    all_chunks: list[str] = []
    page = 0
    next_url: str | None = f"{UNIPROT_BASE}/uniprotkb/search"

    while next_url is not None:
        if page == 0:
            # First request uses params dict; subsequent requests use the full cursor URL
            resp = requests.get(next_url, params=params, timeout=120)
        else:
            resp = requests.get(next_url, timeout=120)

        resp.raise_for_status()
        text = resp.text

        # Strip the TSV header from every page except the first
        if page == 0:
            all_chunks.append(text)
        else:
            # text starts with a header line followed by \n — skip it
            newline_pos = text.find("\n")
            if newline_pos != -1:
                all_chunks.append(text[newline_pos + 1:])

        next_url = _parse_next_link(resp.headers.get("Link"))
        page += 1
        total_so_far = sum(chunk.count("\n") for chunk in all_chunks)
        print(f"  Page {page:>3}  —  ~{total_so_far:,} rows so far", end="\r")

        time.sleep(0.5)  # gentle pacing between pages

    print(f"\nFinished: {page} pages fetched")

    # Write all chunks to disk as a single TSV
    with open(SWISSPROT_PATH, "w", encoding="utf-8") as fh:
        fh.writelines(all_chunks)

    print(f"Saved to {SWISSPROT_PATH}  ({SWISSPROT_PATH.stat().st_size / 1e6:.1f} MB)")

### 1.3 Parse into DataFrame

In [ ]:
# Load the TSV — Polars infers dtypes automatically
raw = pl.read_csv(SWISSPROT_PATH, separator="\t", infer_schema_length=5000)

# Normalise column names: lowercase, strip whitespace, replace spaces/hyphens with underscores
raw.columns = [
    re.sub(r"[\s\-]+", "_", col.strip().lower())
    for col in raw.columns
]

print(f"Shape: {raw.shape}")
raw.head(5)

#### Swiss-Prot DataFrame columns

| Column | Type | Description |
|---|---|---|
| `entry` | `str` | **UniProtKB accession** — the stable, primary identifier for this protein entry (e.g. `P04637`). Accessions never change; old accessions become secondary and remain searchable. |
| `entry_name` | `str` | **Entry name** in `GENE_ORGANISM` format (e.g. `P53_HUMAN`). These are mnemonic but can be updated; use the accession as the stable key. |
| `protein_names` | `str` | **Recommended and alternative protein names**, concatenated. The first name (before any parentheses) is the IUPAC-recommended full name. |
| `gene_names` | `str` | **Gene symbol(s)** associated with the protein, space-separated. The first symbol is the primary name; synonyms and ORF names follow. |
| `organism` | `str` | **Source organism** full name (always `Homo sapiens (Human)` in this dataset). |
| `length` | `i64` | **Sequence length** in amino acids (residues). Reflects the canonical isoform. |
| `mass` | `i64` | **Molecular mass** in Daltons (Da) computed from the canonical sequence, before post-translational modifications. |
| `sequence` | `str` | **Canonical amino-acid sequence** in one-letter code. Non-canonical isoforms are stored separately in the isoforms section. |
| `gene_ontology_ids` | `str` | **GO accessions** (e.g. `GO:0005654`) associated with this entry, semicolon-separated. Covers Biological Process (BP), Molecular Function (MF), and Cellular Component (CC) ontologies. |
| `gene_ontology_(go)` | `str` | **GO term names** with their aspect prefix (e.g. `nucleus [CC]`), semicolon-separated, corresponding to the IDs above. |
| `keywords` | `str` | **Swiss-Prot keywords** — a controlled vocabulary of ~1,200 terms summarising function, disease, PTMs, and other features. Semicolon-separated. |
| `annotated_sequences` | `i64` | **Number of annotated sequence features** (signal peptides, domains, active sites, binding sites, PTM sites, etc.) recorded in the feature table. A proxy for annotation depth. |
| `annotation_score` | `f64` | **UniProt annotation score** (1–5 stars). A heuristic combining the breadth of annotations present; 5 = most comprehensively annotated. Not a quality score per se, but useful for stratifying entries. |
| `reviewed` | `str` | **Review status** — always `reviewed` (Swiss-Prot) in this dataset. Entries in TrEMBL carry `unreviewed`. |